# Fase 7 - Reel Broadcast Instagram (30s, vertical 9:16)

Toma un **broadcast ya generado** (cenital con marcador + posesion + velocidad Kalman + eventos + minimapa + heatmap) y lo formatea **vertical 1080x1920** para Instagram.

- Contenido centrado (blurred-pad, no recorta paneles)
- Barras arriba/abajo con branding (aprovecha el espacio)
- Cards titulo/cierre + color grade. Mudo (musica aparte).

Generalizado: cambia `SOURCE` por cualquier mp4 (broadcast u overlay) y re-corre.

In [1]:
import subprocess
from pathlib import Path

ROOT = Path('/workspace/FutBotMX-UAQTeam')
OUT  = ROOT / 'notebooks/fase_7_visuales/outputs'
TMP  = OUT / '_reel_tmp'
TMP.mkdir(parents=True, exist_ok=True)

# ---- Config (editable) ----
SOURCE = OUT / 'broadcast_IMG_9938_min1.mp4'   # broadcast fuente
FONT   = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'
W, H   = 1080, 1920
START, CONTENT, CARD = 10.0, 25.0, 2.5   # 2.5 + 25 + 2.5 = 30s
FINAL  = OUT / 'reel_broadcast_instagram_30s.mp4'

assert SOURCE.exists(), f'no existe {SOURCE}'

def probe(path, key):
    return subprocess.run(['ffprobe','-v','error','-select_streams','v:0',
        '-show_entries',f'stream={key}','-of','csv=p=0',str(path)],
        capture_output=True,text=True).stdout.strip()

SW, SH = int(probe(SOURCE,'width')), int(probe(SOURCE,'height'))
fg_h = int(round(W * SH / SW / 2) * 2)        # alto del contenido escalado a ancho W
bar  = (H - fg_h) // 2                          # alto de cada barra (espacio muerto)
print(f'fuente {SW}x{SH} -> contenido {W}x{fg_h}, barras {bar}px arriba/abajo')

fuente 848x816 -> contenido 1080x1040, barras 440px arriba/abajo


In [2]:
# Helpers ffmpeg
def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2500:])
        raise RuntimeError('ffmpeg fail')

def dtxt(text, y, size, color='white', alpha=0.6):
    t = text.replace('\\', '').replace(':', '\\:').replace("'", '\u2019')
    return (f"drawtext=fontfile={FONT}:text='{t}':fontcolor={color}:"
            f"fontsize={size}:x=(w-text_w)/2:y={y}:"
            f"box=1:boxcolor=black@{alpha}:boxborderw=16")

def make_card(path, lines):
    vf = f'color=c=0x0a0a0a:s={W}x{H}:d={CARD}:r=30'
    for (txt, y, sz, col) in lines:
        vf += ',' + dtxt(txt, y, sz, col)
    run(['ffmpeg','-y','-f','lavfi','-i',vf,'-frames:v',str(int(CARD*30)),
         '-r','30','-c:v','libx264','-pix_fmt','yuv420p',str(path)])

In [3]:
# Cards
card_t = TMP / 'bc_card_title.mp4'
make_card(card_t, [
    ('FutBotMX 2026', H//2 - 200, 96, 'white'),
    ('UAQ Team  -  Broadcast', H//2 - 50, 60, '0x39d98a'),
    ('Segmentacion - Eventos - Kalman - Minimapa', H//2 + 90, 40, '0xcccccc'),
])
card_o = TMP / 'bc_card_out.mp4'
make_card(card_o, [
    ('SAM 3 + YOLO + ByteTrack + Kalman', H//2 - 110, 50, 'white'),
    ('Marcador - Posesion - Heatmap - Velocidad', H//2, 44, '0x39d98a'),
    ('github.com/RodMed0709/FutBotMX-UAQTeam', H//2 + 150, 36, '0xaaaaaa'),
])
print('cards OK')

cards OK


In [4]:
# Contenido vertical: broadcast centrado + barras con branding
content = TMP / 'bc_content.mp4'
# textos en las barras (espacio muerto arriba/abajo)
top1 = dtxt('FutBotMX 2026  -  UAQ Team', max(40, bar//2 - 70), 48)
top2 = dtxt('Broadcast en vivo', max(120, bar//2 + 10), 38, '0x39d98a')
bot_y = H - bar + max(40, bar//2 - 80)
bot1 = dtxt('Segmentacion + Eventos + Minimapa + Heatmap', bot_y, 38, '0x39d98a')
bot2 = dtxt('Velocidad balon (Kalman)  -  Posesion  -  Marcador', bot_y + 70, 32, '0xdddddd')
fc = (
    f'[0:v]split=2[b][f];'
    f'[b]scale=-2:{H},crop={W}:{H},boxblur=22:2,eq=brightness=-0.22:saturation=0.5[bg];'
    f'[f]scale={W}:{fg_h},eq=contrast=1.05:saturation=1.12[fg];'
    f'[bg][fg]overlay=(W-w)/2:(H-h)/2[v0];'
    f'[v0]{top1}[v1];[v1]{top2}[v2];[v2]{bot1}[v3];[v3]{bot2}[outv]'
)
run(['ffmpeg','-y','-ss',str(START),'-t',str(CONTENT),'-i',str(SOURCE),
     '-filter_complex',fc,'-map','[outv]','-r','30',
     '-c:v','libx264','-pix_fmt','yuv420p',str(content)])
print('contenido OK')

contenido OK


In [5]:
# Concat -> reel final
lst = TMP / 'bc_concat.txt'
lst.write_text('\n'.join(f"file '{p}'" for p in [card_t, content, card_o]) + '\n')
run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(lst),
     '-r','30','-c:v','libx264','-pix_fmt','yuv420p',str(FINAL)])

dur = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
                      '-of','csv=p=0',str(FINAL)],capture_output=True,text=True).stdout.strip()
print(f'\nREEL BROADCAST LISTO -> {FINAL}')
print(f'duracion: {dur}s | {FINAL.stat().st_size/1e6:.1f}MB | {W}x{H} vertical')


REEL BROADCAST LISTO -> /workspace/FutBotMX-UAQTeam/notebooks/fase_7_visuales/outputs/reel_broadcast_instagram_30s.mp4
duracion: 30.000000s | 5.4MB | 1080x1920 vertical
